In [18]:
import pandas as pd
import jieba
from gensim.models import Word2Vec

# 加载餐厅评价数据
df = pd.read_csv("train.csv")
df = df.dropna(subset=["comment"])  # 去除空评论

# 对评论进行结巴分词
df["cut_comment"] = df["comment"].apply(lambda x: list(jieba.cut(x)))
corpus = df["cut_comment"].tolist()

print("✅ 数据加载完成，总评论数：", len(corpus))
print("第一条评论分词结果：", corpus[0])

✅ 数据加载完成，总评论数： 10000
第一条评论分词结果： ['一如既往', '地', '好吃', '，', '希望', '可以', '开', '到', '其他', '城市']


In [23]:
# 训练 Skip-Gram 词向量模型（sg=1 代表 Skip-Gram，必须这样设置）
model = Word2Vec(
    sentences=corpus,
    sg=1,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4
)

# 开始训练
model.train(corpus, total_examples=len(corpus), epochs=10)
model.save("word2vec_skipgram.model")  # 保存模型

print("✅ Skip-Gram 模型训练完成！")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


✅ Skip-Gram 模型训练完成！


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [20]:
# 获取“环境”的词向量
if "环境" in model.wv:
    vec_env = model.wv["环境"]
    print("“环境”的词向量：\n", vec_env)
    print("词向量形状：", vec_env.shape)
else:
    print("⚠️ 语料中未找到“环境”一词")

“环境”的词向量：
 [-0.12793498  0.09288266 -0.27759454 -0.04747113 -0.2837254  -0.31856173
  0.14091153  0.8188092  -0.19815926 -0.02261728 -0.25186297 -0.37236333
  0.3221021   0.04716608 -0.17311479  0.12893106 -0.06048485 -0.09745026
 -0.14139518 -1.1497073  -0.17540985 -0.10717174 -0.05686919 -0.3660011
 -0.6097385   0.25364247 -0.4047006   0.17721103  0.08600662 -0.07928934
  0.32641852  0.3065043   0.08637865 -0.23276174 -0.03448584  0.24170204
  0.2898701   0.09507314  0.18804824 -0.30471188  0.11697765 -0.07595078
 -0.07295196 -0.36541858 -0.0903443  -0.5836546  -0.25441313  0.04819461
  0.31049126 -0.00979766  0.34715697 -0.03096027  0.3407681   0.6222262
  0.10567722  0.5102146  -0.41515687  0.39380342 -0.2531954   0.00870213
  0.04220043 -0.13938886  0.7615151  -0.18792628 -0.11906379  0.6285459
  0.04742416  0.02946887 -0.80648804  0.39887062 -0.38555127 -0.42651471
  0.3302489  -0.4485819   0.70260096 -0.19007294  0.31383553 -0.17968388
  0.10005559  0.14984313 -0.01524723  0.040

In [21]:
# 查找与“好吃”语义最接近的3个词
if "好吃" in model.wv:
    similar_words = model.wv.most_similar("好吃", topn=3)
    print("与“好吃”最接近的3个词：")
    for word, score in similar_words:
        print(f"{word}：{score:.4f}")
else:
    print("⚠️ 语料中未找到“好吃”一词")

与“好吃”最接近的3个词：
极力推荐：0.6490
厚：0.6425
好次：0.6349


In [22]:
# 计算“好吃”与“美味”、“好吃”与“蟑螂”的相似度
if all(word in model.wv for word in ["好吃", "美味", "蟑螂"]):
    sim_good = model.wv.similarity("好吃", "美味")
    sim_bad = model.wv.similarity("好吃", "蟑螂")
    print(f"“好吃”与“美味”的相似度：{sim_good:.4f}")
    print(f"“好吃”与“蟑螂”的相似度：{sim_bad:.4f}")
else:
    print("⚠️ 语料中缺少部分词汇（好吃/美味/蟑螂）")

“好吃”与“美味”的相似度：0.5120
“好吃”与“蟑螂”的相似度：0.1656
